# LeetCode 453 — Minimum Moves to Equal Array Elements bằng Python

Link đề: https://leetcode.com/problems/minimum-moves-to-equal-array-elements/ · Độ khó: **Medium** · Chủ đề: Array, Math

Skeleton LeetCode:

```python
class Solution:
    def minMoves(self, nums: List[int]) -> int:
        ...
```

Bài này nổi tiếng vì **đề nghe rất rắc rối** ("mỗi lượt tăng `n - 1` phần tử") nhưng chỉ cần **đổi góc nhìn** là lời giải gói gọn trong một dòng. Notebook sẽ đi từ mô phỏng thô, đến góc nhìn đổi chiều, đến công thức toán, rồi hai cách làm khác (xếp theo "tầng" và duyệt một lượt).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một lượt ta **chọn một phần tử rồi tăng tất cả phần tử còn lại thêm 1**; cần số lượt **ít nhất** để cả mảng bằng nhau.
2. Nắm các công cụ Python: `min`, `max`, `sum`, `len`, biểu thức sinh (generator expression), list comprehension để "tăng mọi phần tử trừ một vị trí", `sorted`, `enumerate`, và việc Python có số nguyên **không tràn** (khác `int` 32-bit của C++/Java).
3. Đi từ **mô phỏng từng lượt** → **đổi góc nhìn "tăng n-1 phần tử = giảm 1 phần tử"** → **công thức `sum - n*min`** → **xếp tầng sau khi sắp xếp** → **duyệt một lượt, cập nhật min khi chạy**.
4. Hiểu mấu chốt: tăng `n - 1` phần tử thêm 1 **về tương đối** giống hệt việc **giảm một phần tử đi 1**; mọi phần tử cuối cùng phải về mức của **phần tử nhỏ nhất**.
5. Tránh các lỗi kinh điển: dùng `max` thay `min`, nhầm với bài "tăng/giảm một phần tử" (bài 462 dùng **trung vị**), quên nhân `n`, và tràn số 32-bit khi chuyển sang ngôn ngữ khác.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có `n` phần tử. Trong **một lượt (move)** bạn phải **tăng đúng `n - 1` phần tử** của mảng thêm `1` (tức là chọn một phần tử "được nghỉ", mọi phần tử khác cộng 1). Trả về **số lượt ít nhất** để tất cả phần tử **bằng nhau**.

**Ví dụ 1**

- Input: `nums = [1,2,3]`
- Output: `3`
- Giải thích: `[1,2,3] → [2,3,3] → [3,4,3] → [4,4,4]` (mỗi lượt tăng hai phần tử).

**Ví dụ 2**

- Input: `nums = [1,1,1]`
- Output: `0`

**Ràng buộc:**

- `n == nums.length`
- `1 <= nums.length <= 10^5`
- `-10^9 <= nums[i] <= 10^9`
- Đáp án được đảm bảo nằm trong số nguyên **32-bit**.

In [1]:
from typing import List
import random
from collections import deque

examples = [([1, 2, 3], 3), ([1, 1, 1], 0)]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[1, 2, 3] -> kỳ vọng 3
[1, 1, 1] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. `min`, `max`, `sum`, `len` và biểu thức sinh (generator expression)

`min(a)` / `max(a)` trả phần tử nhỏ / lớn nhất; `sum(a)` là tổng; `len(a)` là số phần tử. Biểu thức sinh `sum(x - m for x in a)` là cách viết gọn của vòng `for` cộng dồn, không cần tạo list tạm.

In [2]:
a = [4, 1, 7, 1]
print(min(a), max(a), sum(a), len(a))             # 1 7 13 4
m = min(a)
print(sum(x - m for x in a))                      # (4-1)+(1-1)+(7-1)+(1-1) = 9

1 7 13 4
9


### 3.2. "Tăng mọi phần tử trừ một vị trí" bằng list comprehension

Một lượt của đề: chọn vị trí `skip`, các vị trí khác cộng `1`. Với list comprehension: `[x + 1 if j != skip else x for j, x in enumerate(a)]`. `enumerate(a)` cho ta cả chỉ số `j` lẫn giá trị `x`.

In [3]:
a = [1, 2, 3]
skip = 2                          # vị trí được "nghỉ"
b = [x + 1 if j != skip else x for j, x in enumerate(a)]
print(a, "->", b)                # [1, 2, 3] -> [2, 3, 3]

[1, 2, 3] -> [2, 3, 3]


### 3.3. Dịch cả mảng một đoạn không đổi bản chất

Nếu cộng **cùng một số** `d` vào **mọi** phần tử thì hiệu giữa các phần tử **không đổi**; nên mảng "bằng nhau" hay không cũng không đổi. Đây là chìa khoá: một lượt "tăng `n-1` phần tử" = "tăng **tất cả** thêm 1" rồi "giảm phần tử được nghỉ đi 1". Bước thứ nhất (tăng tất cả) **không làm thay đổi hiệu số**, nên về mặt tương đối chỉ còn bước thứ hai.

In [4]:
a = [1, 2, 3]
skip = 2
cach_de = [x + 1 if j != skip else x for j, x in enumerate(a)]            # đề: tăng n-1 phần tử
cach_doi = [(x + 1) - (1 if j == skip else 0) for j, x in enumerate(a)]    # tăng tất cả rồi giảm 1 phần tử
print(cach_de, cach_doi, cach_de == cach_doi)      # [2, 3, 3] [2, 3, 3] True

# Hiệu giữa các phần tử khi tăng TẤT CẢ thêm 1 không đổi
def diffs(v): return [v[i + 1] - v[i] for i in range(len(v) - 1)]
print(diffs(a), diffs([x + 1 for x in a]))         # [1, 1] [1, 1]

[2, 3, 3] [2, 3, 3] True
[1, 1] [1, 1]


### 3.4. `sorted` và duyệt theo cặp liền kề

`sorted(a)` trả về list mới đã sắp tăng dần (không đổi `a`). Duyệt các cặp liền kề dùng `range(1, len(s))` với `s[i - 1]` và `s[i]`; hoặc `zip(s, s[1:])`.

In [5]:
a = [3, 1, 2]
s = sorted(a)
print(a, s)                                       # [3, 1, 2] [1, 2, 3]
for i in range(1, len(s)):
    print(i, s[i - 1], "->", s[i], "chênh", s[i] - s[i - 1])
print(list(zip(s, s[1:])))                        # [(1, 2), (2, 3)]

[3, 1, 2] [1, 2, 3]
1 1 -> 2 chênh 1
2 2 -> 3 chênh 1
[(1, 2), (2, 3)]


### 3.5. Số nguyên Python không tràn, còn 32-bit thì có

Đề đảm bảo **đáp án** vừa 32-bit, nhưng **tổng** `sum(nums)` có thể lên tới `10^5 * 10^9 = 10^14` — vượt 32-bit. Python tự dùng số nguyên lớn nên không lo; trong C++/Java phải dùng `long`. Ví dụ dưới mô phỏng việc "cắt" xuống 32-bit.

In [6]:
def wrap32(x):
    """Mô phỏng tràn int 32-bit có dấu (như C++/Java)."""
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >= (1 << 31) else x

big = [10 ** 9] * 100000
print(sum(big))                    # 100000000000000  (10^14, Python vẫn đúng)
print(wrap32(sum(big)))            # giá trị sai nếu dùng int 32-bit

100000000000000
276447232


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `[1, 2, 3]`. Mỗi lượt có 3 lựa chọn "ai được nghỉ". Gọi `S = (x1 - min) + (x2 - min) + ...` là **tổng khoảng cách tới phần tử nhỏ nhất**. Thử cho **phần tử lớn nhất nghỉ** mỗi lượt:

| Lượt | Mảng sau lượt | Phần tử nghỉ | `S` |
|---|---|---|---|
| 0 | `[1,2,3]` | — | 0+1+2 = 3 |
| 1 | `[2,3,3]` | `3` | 0+1+1 = 2 |
| 2 | `[3,3,4]` | `3` (một trong hai số 3 lớn nhất) | 0+0+1 = 1 |
| 3 | `[4,4,4]` | `4` | 0 |

Quan sát: **`S` giảm đúng `1` mỗi lượt**, nên số lượt cần thiết **đúng bằng `S` ban đầu** `= 3`.

Vì sao? Khi phần tử lớn nhất nghỉ, mọi phần tử khác tăng 1 — trong đó có cả những phần tử nhỏ nhất, nên khoảng cách từ phần tử lớn nhất tới các phần tử còn lại **thu hẹp thêm 1**. Nói cách khác, về mặt **tương đối** chỉ có phần tử nghỉ "tụt xuống 1".

Vậy câu hỏi quy về: "cần bao nhiêu lần giảm 1 để mọi phần tử về mức nhỏ nhất?" — **phần tử `x` cần giảm `x - min`**, tổng là `sum(x - min)`.

In [7]:
a = [1, 2, 3]
print([x - min(a) for x in a], sum(x - min(a) for x in a))     # [0, 1, 2] 3

[0, 1, 2] 3


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng lượt

**Ý tưởng:** lặp cho tới khi cả mảng bằng nhau: mỗi lượt cho **phần tử lớn nhất** nghỉ, tăng các phần tử còn lại thêm 1, đếm lượt. (Chọn phần tử lớn nhất nghỉ là tham lam hợp lý nhất vì nó **thu hẹp khoảng cách max-min**; mục test sẽ kiểm chứng bằng BFS vét cạn đầy đủ.)

Code `moves_simulate`:

In [8]:
def moves_simulate(nums):
    a = list(nums)                       # sao chép để không phá input
    moves = 0
    while max(a) != min(a):
        skip = a.index(max(a))           # vị trí phần tử lớn nhất nghỉ
        a = [x + 1 if j != skip else x for j, x in enumerate(a)]
        moves += 1
    return moves


for nums, expected in examples:
    print(nums, moves_simulate(nums), "kỳ vọng", expected)
print(moves_simulate([1, 5, 9]))        # 12

[1, 2, 3] 3 kỳ vọng 3
[1, 1, 1] 0 kỳ vọng 0
12


**Độ phức tạp:** mỗi lượt tốn `O(n)`; số lượt bằng chính đáp án `k`. Vậy `Time = O(k · n)`, `Space = O(n)`. Với `nums = [-10^9, 10^9, ...]`, `k` có thể ~`2·10^9`... chạy vô tận so với giới hạn thời gian.

**Nút thắt:** ta **không cần thực sự cập nhật mảng**; chỉ cần biết **tổng số lượt**. Mỗi lượt làm giảm đúng `1` một đại lượng nào đó — nếu gọi tên được đại lượng ấy, ta tính thẳng được.

## 6. Ý tưởng thuật toán / bước đột phá

**Đổi góc nhìn:** "tăng `n-1` phần tử thêm 1" khi xét **hiệu giữa các phần tử** giống hệt "**giảm phần tử còn lại đi 1**" (mục 3.3). Ta được bài mới tương đương:

> Mỗi lượt **giảm đúng một phần tử đi 1**. Cần ít lượt nhất để cả mảng bằng nhau.

Với bài mới này:

- Giá trị chung cuối cùng không thể **cao hơn `min(nums)`**: vì ta chỉ giảm mà phần tử nhỏ nhất không thể tăng.
- Giảm quá `min` chỉ thêm lượt dư (mỗi phần tử đã là `min` lại phải bị giảm thêm) nên **tối ưu là hạ mọi phần tử về đúng `min`**.
- Phần tử `x` cần `x - min` lượt giảm. Tổng: **`k = sum(x - min) = sum(nums) - n * min`**.

Kiểm chứng bằng phương trình ban đầu: gọi `k` là số lượt, `x` là giá trị chung cuối. Mỗi lượt tổng tăng `n - 1` nên `s + (n-1)k = n·x`; phần tử nhỏ nhất được tăng `k` lần trừ những lượt nó được nghỉ... trong lời giải tối ưu nó **không bao giờ nghỉ**: `x = min + k`. Thế vào: `s + (n-1)k = n·min + n·k` ⇒ `k = s - n·min`. Hai con đường ra cùng đáp án.

In [9]:
a = [1, 2, 3]
s, n, mn = sum(a), len(a), min(a)
k = s - n * mn
x = mn + k
print(k, x, s + (n - 1) * k == n * x)       # 3 4 True

3 4 True


## 7. Các cách tiếp theo

Tổng cộng 5 cách: (1) mô phỏng, (2) đổi chiều "giảm về min" có vòng lặp, (3) công thức tổng, (4) xếp tầng sau khi sắp xếp, (5) duyệt một lượt theo dõi min khi chạy.

### Cách 2 — Đổi chiều: mỗi phần tử giảm về `min`

**Giải thích từng đoạn:**

1. Tìm `mn = min(nums)`.
2. Với từng phần tử `x`, cần `x - mn` lượt giảm; cộng dồn vào `total`.

Không cần mảng phụ, nhưng vẫn là hai lượt duyệt (một để tìm min, một để cộng).

In [10]:
def moves_to_min(nums):
    mn = min(nums)                # đoạn 1: tìm min
    total = 0
    for x in nums:                # đoạn 2: x cần x - mn lượt giảm
        total += x - mn
    return total


for nums, expected in examples:
    print(nums, moves_to_min(nums), "kỳ vọng", expected)

[1, 2, 3] 3 kỳ vọng 3
[1, 1, 1] 0 kỳ vọng 0


In [11]:
# Code nộp LeetCode — Cách 2
class Solution2:
    def minMoves(self, nums: List[int]) -> int:
        mn = min(nums)
        total = 0
        for x in nums:
            total += x - mn
        return total


print(Solution2().minMoves([1, 2, 3]), Solution2().minMoves([1, 1, 1]))

3 0


**Độ phức tạp Cách 2:** `Time = O(n)`, `Space = O(1)`.

### Cách 3 — Công thức toán `sum(nums) - n * min(nums)`

**Giải thích:** `sum(x - mn) = sum(x) - n·mn` vì `mn` bị trừ đúng `n` lần. Hai hàm dựng sẵn `sum`, `min` làm gọn thành một dòng. Cách này là lời giải "chuẩn" của editorial.

In [12]:
def moves_formula(nums):
    return sum(nums) - len(nums) * min(nums)


for nums, expected in examples:
    print(nums, moves_formula(nums), "kỳ vọng", expected)

[1, 2, 3] 3 kỳ vọng 3
[1, 1, 1] 0 kỳ vọng 0


In [13]:
# Code nộp LeetCode — Cách 3
class Solution3:
    def minMoves(self, nums: List[int]) -> int:
        return sum(nums) - min(nums) * len(nums)


print(Solution3().minMoves([1, 2, 3]), Solution3().minMoves([5]))

3 0


**Độ phức tạp Cách 3:** `Time = O(n)` (hai lượt: `sum` và `min`), `Space = O(1)`.

### Cách 4 — Xếp "tầng" sau khi sắp xếp

**Ý tưởng khác hẳn:** sắp tăng dần `a[0] <= a[1] <= ... <= a[n-1]`. Hình dung các cột chiều cao `a[i]`. Phải hạ các cột xuống mức `a[0]`. Từ mức `a[i-1]` lên mức `a[i]` là một "tầng" dày `a[i] - a[i-1]`; **tầng này bị `i` cột** (các cột `i, i+1, …, n-1`)... đếm kỹ: tầng giữa `a[i-1]` và `a[i]` bị cắt bởi **mọi cột có chỉ số `>= i`**, tức `n - i` cột. Cộng theo cách này cho `sum((a[i]-a[i-1]) * (n - i))`.

Nhưng cũng đếm từ phía dưới: **cột `a[i]`** phải hạ `a[i] - a[0] = (a[i]-a[i-1]) + ... + (a[1]-a[0])`; tầng `(a[j]-a[j-1])` góp vào mọi cột `i >= j`. Hai cách đếm cho cùng tổng. Ta chọn đếm theo **tầng**: tầng giữa `a[i-1]` và `a[i]` bị tính `n - i` lần.

In [14]:
def moves_layers(nums):
    a = sorted(nums)
    n = len(a)
    total = 0
    for i in range(1, n):
        layer = a[i] - a[i - 1]          # độ dày tầng thứ i
        total += layer * (n - i)         # có n - i cột (chỉ số i..n-1) phải hạ qua tầng này
    return total


for nums, expected in examples:
    print(nums, moves_layers(nums), "kỳ vọng", expected)
print(moves_layers([1, 5, 9]))           # 12

[1, 2, 3] 3 kỳ vọng 3
[1, 1, 1] 0 kỳ vọng 0
12


In [15]:
# Code nộp LeetCode — Cách 4
class Solution4:
    def minMoves(self, nums: List[int]) -> int:
        a = sorted(nums)
        n = len(a)
        return sum((a[i] - a[i - 1]) * (n - i) for i in range(1, n))


print(Solution4().minMoves([1, 2, 3]), Solution4().minMoves([1, 1, 1]), Solution4().minMoves([7]))

3 0 0


**Độ phức tạp Cách 4:** sắp xếp `O(n log n)`, duyệt `O(n)`, `Space = O(n)` cho bản sao sắp xếp. Chậm hơn công thức, nhưng giúp **nhìn ra cấu trúc**, và khớp với bài 462 (cùng sắp xếp).

### Cách 5 — Duyệt một lượt, cập nhật `min` khi đang chạy (online)

**Ý tưởng:** không cần biết min toàn cục trước. Giữ `mn` = min của các phần tử **đã thấy** và `ans` = đáp án cho phần đã thấy.

- Gặp `x >= mn`: `x` cần thêm `x - mn` lượt: `ans += x - mn`.
- Gặp `x < mn` (min mới): **tất cả `i` phần tử đã thấy** đều phải hạ thêm `mn - x`: `ans += (mn - x) * i`, rồi `mn = x`. (`x` mới không cần hạ.)

Nhờ vậy xử lý được cả dữ liệu đến từng phần tử một (streaming).

In [16]:
def moves_online(nums):
    mn = nums[0]
    ans = 0
    for i, x in enumerate(nums):        # i = số phần tử đã đứng trước x
        if x < mn:
            ans += (mn - x) * i         # i phần tử cũ phải hạ thêm (mn - x)
            mn = x
        else:
            ans += x - mn
    return ans


for nums, expected in examples:
    print(nums, moves_online(nums), "kỳ vọng", expected)
print(moves_online([3, 2, 1]))          # 3

[1, 2, 3] 3 kỳ vọng 3
[1, 1, 1] 0 kỳ vọng 0
3


In [17]:
# Code nộp LeetCode — Cách 5
class Solution5:
    def minMoves(self, nums: List[int]) -> int:
        mn = nums[0]
        ans = 0
        for i, x in enumerate(nums):
            if x < mn:
                ans += (mn - x) * i
                mn = x
            else:
                ans += x - mn
        return ans


print(Solution5().minMoves([1, 2, 3]), Solution5().minMoves([3, 2, 1]), Solution5().minMoves([4]))

3 3 0


**Độ phức tạp Cách 5:** `Time = O(n)` (đúng một lượt), `Space = O(1)`.

**Tóm tắt:** Cách 1 mô phỏng chậm; Cách 2/3 là lời giải chuẩn; Cách 4 minh hoạ cấu trúc "tầng"; Cách 5 một lượt, hợp khi dữ liệu đến dần.

## 8. TRACE từng bước

Hàm `trace_simulate` in mảng sau từng lượt mô phỏng; `trace_online` in bảng trạng thái của Cách 5.

In [18]:
def trace_simulate(nums):
    a = list(nums)
    moves = 0
    print("bắt đầu:", a)
    while max(a) != min(a):
        skip = a.index(max(a))
        a = [x + 1 if j != skip else x for j, x in enumerate(a)]
        moves += 1
        print(f"lượt {moves}: nghỉ vị trí {skip} -> {a}   (max-min = {max(a) - min(a)})")
    print("tổng lượt:", moves)
    return moves


trace_simulate([1, 2, 3])

bắt đầu: [1, 2, 3]
lượt 1: nghỉ vị trí 2 -> [2, 3, 3]   (max-min = 1)
lượt 2: nghỉ vị trí 1 -> [3, 3, 4]   (max-min = 1)
lượt 3: nghỉ vị trí 2 -> [4, 4, 4]   (max-min = 0)
tổng lượt: 3


3

In [19]:
trace_simulate([1, 3, 6])

bắt đầu: [1, 3, 6]
lượt 1: nghỉ vị trí 2 -> [2, 4, 6]   (max-min = 4)
lượt 2: nghỉ vị trí 2 -> [3, 5, 6]   (max-min = 3)
lượt 3: nghỉ vị trí 2 -> [4, 6, 6]   (max-min = 2)
lượt 4: nghỉ vị trí 1 -> [5, 6, 7]   (max-min = 2)
lượt 5: nghỉ vị trí 2 -> [6, 7, 7]   (max-min = 1)
lượt 6: nghỉ vị trí 1 -> [7, 7, 8]   (max-min = 1)
lượt 7: nghỉ vị trí 2 -> [8, 8, 8]   (max-min = 0)
tổng lượt: 7


7

In [20]:
def trace_online(nums):
    mn = nums[0]
    ans = 0
    print(f"{'i':>2} {'x':>3} {'mn trước':>9} {'nhánh':>10} {'cộng thêm':>10} {'ans':>5} {'mn sau':>7}")
    for i, x in enumerate(nums):
        before = mn
        if x < mn:
            add = (mn - x) * i
            ans += add
            mn = x
            branch = "min mới"
        else:
            add = x - mn
            ans += add
            branch = "x >= mn"
        print(f"{i:>2} {x:>3} {before:>9} {branch:>10} {add:>10} {ans:>5} {mn:>7}")
    return ans


print(trace_online([4, 2, 5, 1, 3]))
print("kiểm tra công thức:", sum([4, 2, 5, 1, 3]) - 5 * 1)

 i   x  mn trước      nhánh  cộng thêm   ans  mn sau
 0   4         4    x >= mn          0     0       4
 1   2         4    min mới          2     2       2
 2   5         2    x >= mn          3     5       2
 3   1         2    min mới          3     8       1
 4   3         1    x >= mn          2    10       1
10
kiểm tra công thức: 10


Giải nghĩa bảng với `[4, 2, 5, 1, 3]`:

| i | x | Điều gì xảy ra | ans |
|---|---|---|---|
| 0 | 4 | `x >= mn` (mn = 4), cộng 0 | 0 |
| 1 | 2 | min mới: 1 phần tử cũ (4) hạ thêm `4-2 = 2` | 2 |
| 2 | 5 | `5 - 2 = 3` | 5 |
| 3 | 1 | min mới: 3 phần tử cũ hạ thêm `2-1 = 1` mỗi cái = 3 | 8 |
| 4 | 3 | `3 - 1 = 2` | 10 |

Đáp án cuối `10 = (4+2+5+1+3) - 5·1`. ✓

## 9. Code nộp LeetCode

Chọn công thức (Cách 3) làm bản nộp chính: ngắn nhất, `O(n)`, `O(1)`.

In [21]:
class Solution:
    def minMoves(self, nums: List[int]) -> int:
        return sum(nums) - min(nums) * len(nums)


for nums, expected in examples:
    assert Solution().minMoves(nums) == expected
print("OK các ví dụ đề")

OK các ví dụ đề


## 10. Vì sao thuật toán đúng

Đặt `S = Σ (x_i - min)` (tổng khoảng cách tới phần tử nhỏ nhất). Ban đầu `S0 = sum - n·min`; khi mọi phần tử bằng nhau thì `S = 0`.

**Bước 0 — đổi góc nhìn.** Một lượt "tăng mọi phần tử trừ `p`" rồi trừ `1` khỏi **tất cả** phần tử cho ra mảng: `x_p - 1` còn các phần tử khác giữ nguyên. Phép trừ chung này không đổi `S` và không đổi việc "bằng nhau", nên **một lượt ≡ giảm đúng một phần tử đi 1**.

**Bước 1 — cận dưới `k ≥ S0`.** Giảm một phần tử `x_p` đi 1 làm `S` thay đổi thế nào?

- Nếu `x_p > min`: min không đổi, chỉ `x_p - min` giảm 1 ⇒ `S` giảm đúng 1.
- Nếu `x_p = min` (min giảm xuống `min - 1`): mọi phần tử khác cách min mới xa hơn 1 ⇒ `S` **không giảm** (tăng thêm `n - 1` hoặc, với `n = 1`, giữ nguyên).

Vậy mỗi lượt `S` giảm **nhiều nhất 1**. Vì `S` phải từ `S0` về `0`, cần ít nhất `S0` lượt.

**Bước 2 — cận trên `k ≤ S0` (đạt được).** Mỗi lượt giảm một phần tử đang lớn hơn min (tương ứng "cho phần tử lớn nhất nghỉ" trong đề gốc). Khi chưa bằng nhau luôn có phần tử như vậy; mỗi lượt `S` giảm đúng 1 nên sau đúng `S0` lượt thì `S = 0`.

**Kết luận:** `k = S0 = sum - n·min`. Các cách 2–5 đều tính cùng đại lượng `S0` (Cách 4 qua tầng, Cách 5 qua cập nhật min khi chạy); Cách 1 mô phỏng đúng chiến lược ở Bước 2.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng | `O(k · n)` | `O(n)` | `k` là đáp án, có thể rất lớn |
| 2. Giảm về min (vòng lặp) | `O(n)` | `O(1)` | hai lượt duyệt |
| 3. Công thức | `O(n)` | `O(1)` | ngắn nhất |
| 4. Xếp tầng (sort) | `O(n log n)` | `O(n)` | sắp xếp chi phối |
| 5. Duyệt một lượt | `O(n)` | `O(1)` | một lượt, online |

Với `n = 10^5`: Cách 2, 3, 5 chạy trong vài chục mili-giây. Cách 1 không thể chạy khi giá trị lớn.

## 12. Bộ test

### 12.1. Ví dụ đề + edge case

Đối chiếu **tất cả** cách với nhau bằng `assert`.

In [22]:
solutions = {
    "simulate": moves_simulate,
    "to_min": moves_to_min,
    "formula": moves_formula,
    "layers": moves_layers,
    "online": moves_online,
    "Solution": lambda nums: Solution().minMoves(list(nums)),
}

edge = [
    ([1, 2, 3], 3),
    ([1, 1, 1], 0),
    ([5], 0),                         # n = 1
    ([2, 2], 0),
    ([1, 2], 1),                      # n = 2
    ([-3, 0, 3], 9),                  # số âm
    ([0, 0, 0, 0, 7], 7),
    ([3, 2, 1], 3),                   # giảm dần
    ([1, 100], 99),
]
for nums, expected in edge:
    for name, fn in solutions.items():
        if name == "simulate" and max(nums) - min(nums) > 200:
            continue
        got = fn(nums)
        assert got == expected, (name, nums, got, expected)
print("OK", len(edge), "ca, mọi cách đều khớp")

OK 9 ca, mọi cách đều khớp


### 12.2. Phản ví dụ cho các cách làm sai

Các công thức "nghe có vẻ đúng" nhưng sai; kiểm chứng bằng `[1, 2, 3]` (đáp án đúng 3) và `[1, 5, 9]` (đáp án đúng 12).

In [23]:
def wrong_max(nums):           # sai: tính về max
    return sum(max(nums) - x for x in nums)

def wrong_sum_minus_min(nums): # sai: quên nhân n
    return sum(nums) - min(nums)

def wrong_range(nums):         # sai: chỉ lấy max - min
    return max(nums) - min(nums)

for nums in ([1, 2, 3], [1, 5, 9], [3, 2, 1]):
    print(nums, "đúng", moves_formula(nums),
          "| theo max", wrong_max(nums),
          "| sum-min", wrong_sum_minus_min(nums),
          "| max-min", wrong_range(nums))

[1, 2, 3] đúng 3 | theo max 3 | sum-min 5 | max-min 2
[1, 5, 9] đúng 12 | theo max 12 | sum-min 14 | max-min 8
[3, 2, 1] đúng 3 | theo max 3 | sum-min 5 | max-min 2


`wrong_max` trùng đáp án ở các mảng **đối xứng** như `[1, 5, 9]` nên dễ lọt; `max - min` sai ngay khi có phần tử "giữa" hoặc lặp. Phản ví dụ rõ nhất: `[1, 1, 9]` đúng là `8`, nhưng "theo max" cho `16`.

In [24]:
print(moves_formula([1, 1, 9]), wrong_max([1, 1, 9]))     # 8 16
assert moves_formula([1, 1, 9]) != wrong_max([1, 1, 9])

8 16


### 12.3. Oracle độc lập: BFS vét cạn trên mảng nhỏ

BFS thử **mọi** cách chọn phần tử nghỉ ở mỗi lượt (không giả sử chiến lược tham lam). Vì cộng cùng hằng vào mọi phần tử không đổi tính "bằng nhau", ta **chuẩn hoá** trạng thái bằng cách trừ đi `min` để không gian trạng thái hữu hạn.

In [25]:
def moves_bfs(nums):
    start = tuple(nums)
    def norm(t):
        m = min(t)
        return tuple(x - m for x in t)
    start = norm(start)
    if len(set(start)) == 1:
        return 0
    seen = {start}
    q = deque([(start, 0)])
    while q:
        state, d = q.popleft()
        for skip in range(len(state)):
            nxt = norm(tuple(x + 1 if j != skip else x for j, x in enumerate(state)))
            if len(set(nxt)) == 1:
                return d + 1
            if nxt not in seen:
                seen.add(nxt)
                q.append((nxt, d + 1))


print(moves_bfs([1, 2, 3]), moves_bfs([1, 1, 1]), moves_bfs([1, 5, 9]))

3 0 12


In [26]:
# So BFS với công thức trên MỌI mảng độ dài 1..4, giá trị 0..4
import itertools
count = 0
for n in range(1, 5):
    for tup in itertools.product(range(5), repeat=n):
        exp = moves_bfs(list(tup))
        assert exp == moves_formula(list(tup)) == moves_simulate(list(tup)) == moves_online(list(tup)) == moves_layers(list(tup)), tup
        count += 1
print("đã so", count, "mảng với BFS: khớp hoàn toàn")

đã so 780 mảng với BFS: khớp hoàn toàn


### 12.4. Ngẫu nhiên có seed, đúng ràng buộc

Sinh mảng ngẫu nhiên (seed cố định); đối chiếu các cách `O(n)` và cách sắp xếp. Với mảng rất lớn kiểm tra thêm tính đáp án vừa 32-bit khi chọn giá trị hẹp.

In [27]:
random.seed(453)
fast = [moves_to_min, moves_formula, moves_layers, moves_online, solutions["Solution"]]

for _ in range(500):
    n = random.randint(1, 60)
    lo = random.choice([-10 ** 9, -50, 0])
    hi = lo + random.choice([5, 1000, 10 ** 6])
    hi = min(hi, 10 ** 9)
    nums = [random.randint(lo, hi) for _ in range(n)]
    results = {fn(nums) for fn in fast}
    assert len(results) == 1, (nums, results)

# mảng lớn n = 10^5, giá trị hẹp để đáp án vừa 32-bit
big = [random.randint(-10000, 10000) for _ in range(10 ** 5)]
vals = {fn(big) for fn in fast}
assert len(vals) == 1
ans = vals.pop()
print("đáp án mảng lớn:", ans, "vừa 32-bit:", -2 ** 31 <= ans < 2 ** 31)

# so với mô phỏng ở mảng nhỏ giá trị bé
for _ in range(200):
    nums = [random.randint(0, 6) for _ in range(random.randint(1, 6))]
    assert moves_simulate(nums) == moves_formula(nums)
print("OK ngẫu nhiên")

đáp án mảng lớn: 998655605 vừa 32-bit: True
OK ngẫu nhiên


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tính khoảng cách về `max` thay vì `min`

Nghĩ "tăng cho bằng phần tử lớn nhất". Nhưng mỗi lượt cái "bằng" cũng bị tăng theo, nên mục tiêu không đứng yên.

In [28]:
def sai_1(nums):
    return sum(max(nums) - x for x in nums)

print(sai_1([1, 1, 9]), "đúng phải là", moves_formula([1, 1, 9]))      # 16 vs 8
assert sai_1([1, 1, 9]) != moves_formula([1, 1, 9])
print(sai_1([1, 2, 9]), "đúng phải là", moves_formula([1, 2, 9]))      # 15 vs 9

16 đúng phải là 8
15 đúng phải là 9


**Vì sao sai:** mỗi lượt phần tử "đích" cũng bị tăng theo nên mục tiêu không đứng yên; chỉ có phần tử nhỏ nhất là mốc bất động (không bao giờ phải giảm xuống dưới nó). `[1,1,9]` theo max cho `8+8+0 = 16`, nhưng thực tế chỉ cần `8` lượt (về tương đối: hạ `9` xuống `1`).

**Đúng:** dùng `min`: `sum(nums) - n * min(nums)`.

### Lỗi 2: Quên nhân `n` với `min`

`sum(nums) - min(nums)` chỉ đúng khi `n = 1` hay `n = 2` đặc biệt (với `n = 1`, `sum - min = 0`).

In [29]:
def sai_2(nums):
    return sum(nums) - min(nums)

print(sai_2([1, 2, 3]), "đúng phải là", moves_formula([1, 2, 3]))   # 5 vs 3

5 đúng phải là 3


**Đúng:** `min(nums) * len(nums)` — mỗi phần tử đều phải trừ `min` một lần.

### Lỗi 3: Nhầm với bài "tăng hoặc giảm một phần tử" (LeetCode 462)

Bài 462 cho phép **tăng/giảm một phần tử đi 1** → đích là **trung vị**, không phải `min`. Ở bài này (tăng `n-1` phần tử) đích là `min`.

In [30]:
def minmoves_462(nums):
    a = sorted(nums)
    med = a[len(a) // 2]
    return sum(abs(x - med) for x in a)

print(moves_formula([1, 10, 2, 9]), minmoves_462([1, 10, 2, 9]))     # 18 vs 16

18 16


**Đúng:** đọc kỹ thao tác của đề: "tăng **n-1** phần tử" khác "tăng/giảm **một** phần tử".

### Lỗi 4: Mô phỏng thật rồi bị "Time Limit Exceeded"

Mô phỏng từng lượt chạy `O(k·n)`; với `[1, 10^9]` cần `10^9 - 1` lượt.

In [31]:
import time
t0 = time.perf_counter()
print(moves_simulate([1, 3000]))                 # 2999 lượt, vẫn còn nhanh
print(f"mô phỏng [1, 3000] mất {time.perf_counter() - t0:.4f}s")
t0 = time.perf_counter()
print(moves_formula([1, 10 ** 9]))
print(f"công thức [1, 10^9] mất {time.perf_counter() - t0:.6f}s")

2999
mô phỏng [1, 3000] mất 0.0011s
999999999
công thức [1, 10^9] mất 0.000049s


**Đúng:** biến việc "đếm lượt" thành "tính công thức".

### Lỗi 5 (khi chuyển sang ngôn ngữ khác): tràn `int` 32-bit

Trong Python không lo, nhưng bản Java/C++ phải dùng `long` cho tổng. Mô phỏng tràn:

In [32]:
big = [10 ** 9] * 100000
print("tổng thật:", sum(big), "| nếu cắt 32-bit:", wrap32(sum(big)))

tổng thật: 100000000000000 | nếu cắt 32-bit: 276447232


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "mỗi lượt tăng/giảm **tất cả trừ một**" → nghĩ **đổi góc nhìn**: tương đương thay đổi **một** phần tử theo chiều ngược lại.
- Thấy "mọi phần tử bằng nhau" mà thao tác **chung** (tác động cả nhóm) → kiểm tra **hiệu số** (bất biến dưới việc dịch cả nhóm).
- Cụm "đáp án được đảm bảo vừa 32-bit" là gợi ý **trung gian có thể tràn** (tổng).
- Cách kiểm tra nhanh: thử `[1,2,3]` (đáp án 3) và `[1,1,9]` (đáp án 8).
- Cùng họ "đưa mảng về bằng nhau": 462 (tăng/giảm một phần tử → trung vị), 1658, 2448 (chi phí có trọng số → trung vị có trọng số).

## 15. Biến thể của bài

1. **Giảm `n-1` phần tử mỗi lượt** (thay vì tăng): đối xứng, đưa mọi phần tử về `max`: `n*max - sum`.
2. **Tăng `k` phần tử mỗi lượt** (`1 <= k < n`): bài khó hơn (cần điều kiện chia hết); nghĩ tới tổng và nhóm.
3. **Tăng/giảm một phần tử mỗi lượt** (LeetCode 462): đích là trung vị.
4. **Hỏi giá trị chung cuối cùng** thay vì số lượt: `min + k`.

Dưới đây là biến thể 1 và 4, có test.

In [33]:
def moves_decrement_n_minus_1(nums):
    """Mỗi lượt GIẢM n-1 phần tử đi 1: về max."""
    return len(nums) * max(nums) - sum(nums)

def final_value(nums):
    """Giá trị chung cuối cùng sau số lượt tối ưu (bài 453)."""
    return min(nums) + moves_formula(nums)

print(moves_decrement_n_minus_1([1, 2, 3]))       # 3
print(final_value([1, 2, 3]))                     # 4
assert moves_decrement_n_minus_1([1, 2, 3]) == moves_formula([-1, -2, -3])   # đối xứng qua phép đổi dấu
print("OK biến thể")

3
4
OK biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | tăng thật, cho max nghỉ | `O(k·n)` | `O(n)` | chỉ để kiểm chứng trên mảng nhỏ |
| 2. Giảm về min | `Σ(x - min)` bằng vòng lặp | `O(n)` | `O(1)` | khi muốn code rõ ràng, dễ giải thích |
| 3. Công thức | `sum - n·min` | `O(n)` | `O(1)` | **nộp bài**, ngắn gọn |
| 4. Xếp tầng | `Σ(a[i]-a[i-1])·(n-i)` | `O(n log n)` | `O(n)` | hiểu cấu trúc; liên hệ bài 462 |
| 5. Một lượt online | cập nhật min khi chạy | `O(n)` | `O(1)` | dữ liệu đến dần (stream), không giữ mảng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Solution().minMoves([2, 2, 2])`, `Solution().minMoves([1, 3])`, `Solution().minMoves([4, 1, 7])`, `moves_layers([1, 4, 4, 10])`. Điền vào các biến `ans1_a, ans1_b, ans1_c, ans1_d` (hiện là `None`).

**Bài 2 (tính tay).** Với `nums = [5, 3, 8, 3]`, tính `sum`, `min`, `n` và số lượt. Điền `ans2`.

**Bài 3 (điền TODO).** Hoàn thành `moves_todo(nums)` theo Cách 5 (duyệt một lượt).

In [34]:
ans1_a = ans1_b = ans1_c = ans1_d = None      # điền sau khi đoán
ans2 = None                                    # điền sau khi tính tay

def moves_todo(nums):
    # TODO 1: mn = nums[0]; ans = 0
    # TODO 2: for i, x in enumerate(nums):
    # TODO 3:     nếu x < mn: ans += (mn - x) * i; mn = x
    # TODO 4:     ngược lại: ans += x - mn
    # TODO 5: return ans
    pass

# KHÔNG gọi moves_todo ở đây để Run All không lỗi.

**Bài 4 (tự viết lại).** Viết `my_min_moves(nums)` **không dùng `sum` và `min` dựng sẵn** (chỉ vòng `for`), trả cùng kết quả.

**Bài 5 (biến thể / mini challenge).** (a) Viết `min_moves_k(nums, k)`: mỗi lượt tăng đúng **`k` phần tử** (với `1 <= k <= n-1`). Gợi ý: nghĩ tới tổng mỗi lượt tăng `k`, hãy tìm đáp án bằng BFS nhỏ để đoán quy luật. (b) Viết `min_moves_to_max(nums)` cho trường hợp **giảm** `n-1` phần tử mỗi lượt và trả về giá trị chung cuối cùng.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: dùng `sum - n*min`; với `[1,4,4,10]` hãy tính từng tầng: `3·3 + 0·2 + 6·1 = 15`. Bài 2: `sum = 19`, `n = 4`, `min = 3`.
- **Mức 2 (vừa):** Bài 3: `i` là số phần tử đứng trước; khi có min mới, nhân `(mn - x)` với `i`. Bài 4: một vòng tìm min, một vòng cộng, thay `sum`/`min` bằng biến tự cập nhật.
- **Mức 3 (sâu):** Bài 5(a): khi `k = n-1` là bài gốc. Với `k < n-1`, thử BFS với mảng nhỏ rồi quan sát điều kiện chia hết của hiệu; không phải mảng nào cũng có đáp án (trả `-1`). Bài 5(b): đối xứng (đổi dấu mảng): số lượt là `n*max - sum`; mỗi lượt tổng giảm `n-1` nên giá trị chung cuối là `max - lượt`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [35]:
# Bài 1
ans1_a, ans1_b, ans1_c, ans1_d = 0, 2, 9, 15
assert ans1_a == Solution().minMoves([2, 2, 2])
assert ans1_b == Solution().minMoves([1, 3])
assert ans1_c == Solution().minMoves([4, 1, 7])
assert ans1_d == moves_layers([1, 4, 4, 10])

# Bài 2: [5,3,8,3] -> sum 19, n 4, min 3 -> 19 - 12 = 7
ans2 = 7
assert ans2 == Solution().minMoves([5, 3, 8, 3])

# Bài 3
def moves_todo(nums):
    mn = nums[0]
    ans = 0
    for i, x in enumerate(nums):
        if x < mn:
            ans += (mn - x) * i
            mn = x
        else:
            ans += x - mn
    return ans

# Bài 4
def my_min_moves(nums):
    mn = nums[0]
    for x in nums:
        if x < mn:
            mn = x
    total = 0
    for x in nums:
        total += x - mn
    return total

for nums in ([1, 2, 3], [5, 3, 8, 3], [7], [-4, 0, 9, -4]):
    assert moves_todo(nums) == my_min_moves(nums) == Solution().minMoves(nums)
print("OK bài 1-4")

OK bài 1-4


In [36]:
# Bài 5(a): tăng đúng k phần tử mỗi lượt, BFS trên trạng thái chuẩn hoá (nhỏ), -1 nếu không thể
from itertools import combinations

def min_moves_k(nums, k):
    n = len(nums)
    def norm(t):
        m = min(t)
        return tuple(x - m for x in t)
    start = norm(tuple(nums))
    if len(set(start)) == 1:
        return 0
    seen = {start}
    q = deque([(start, 0)])
    limit = 40                     # chặn độ sâu cho bản mini (đủ cho mảng nhỏ)
    while q:
        state, d = q.popleft()
        if d >= limit:
            continue
        for chosen in combinations(range(n), k):
            cs = set(chosen)
            nxt = norm(tuple(x + 1 if j in cs else x for j, x in enumerate(state)))
            if len(set(nxt)) == 1:
                return d + 1
            if nxt not in seen and max(nxt) <= 12:
                seen.add(nxt)
                q.append((nxt, d + 1))
    return -1

assert min_moves_k([1, 2, 3], 2) == 3          # k = n-1 là bài gốc
assert min_moves_k([1, 1, 1], 2) == 0
for nums in ([1, 2, 3], [1, 1, 4], [0, 2, 5]):
    assert min_moves_k(nums, len(nums) - 1) == Solution().minMoves(nums)

# Bài 5(b): giảm n-1 phần tử mỗi lượt, về max; giá trị cuối = max - lượt
def min_moves_to_max(nums):
    moves = len(nums) * max(nums) - sum(nums)
    return moves, max(nums) - moves

assert min_moves_to_max([1, 2, 3]) == (3, 0)
assert min_moves_to_max([5]) == (0, 5)
print("OK bài 5")

OK bài 5


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đổi góc nhìn: tăng `n-1` phần tử = giảm 1 phần tử → về `min`)

```python
class Solution:
    def minMoves(self, nums: List[int]) -> int:
        return sum(nums) - min(nums) * len(nums)
```

**Checklist**

- [ ] Hiểu thao tác "tăng `n-1` phần tử" **tương đương** "giảm một phần tử" về mặt hiệu số?
- [ ] Đích là **`min`** (không phải `max`, không phải trung vị)?
- [ ] Công thức: `sum - n * min`, có nhân `n`?
- [ ] Đã thử `[1,2,3]` (3), `[1,1,1]` (0), `[5]` (0), `[1,1,9]` (8)?
- [ ] Có nghĩ tới tràn 32-bit khi tính tổng (Python thì an toàn)?
- [ ] Skeleton đúng `class Solution` / `def minMoves(self, nums: List[int]) -> int`?